# Interpretarea si Explicabilitatea Modelelor

Bine ai venit la tema despre Interpretarea si Explicabilitatea Modelelor.

Construirea unui model care generalizeaza bine reprezinta doar jumatate din munca. Cealalta jumatate este sa poti **explica** ce a invatat modelul: care caracteristici ii influenteaza predictiile, cat contribuie fiecare caracteristica la un anumit rezultat si daca rationamentul modelului este aliniat cu cunostintele din domeniu.

In aceasta tema vei aplica o progresie de tehnici de interpretare: de la cele simple (citirea coeficientilor unui model liniar) pana la cele avansate (valorile SHAP Shapley si explicatiile locale LIME). Vei lucra atat cu explicatii globale (ce a invatat modelul in ansamblu), cat si cu explicatii locale (de ce a fost facuta *aceasta predictie anume*?).

**Ce vei face in aceasta tema**

* Interpretezi **coeficientii modelelor liniare** si intelegi impactul caracteristicilor.
* Vizualizezi si extragi **reguli de decizie din modele de tip arbore**.
* Compari **importanta prin permutare, SHAP si LIME** intre modele.
* Creezi grafice SHAP de tip **summary, dependence si force**.
* Implementezi **importanta prin permutare de la zero**.
* Construiesti un **raport complet de interpretare**.

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

*   Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

*   In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul solutiei. **Nu adauga si nu modifica niciun cod in afara acestor comentarii**.

*   Poti adauga celule noi pentru experimente, dar evaluatorul le va ignora, asa ca nu te baza pe celulele create de tine pentru codul de raspuns; foloseste spatiile oferite in notebook.

*   Evita folosirea variabilelor globale, cu exceptia situatiilor absolut necesare. Evaluatorul iti testeaza codul intr-un mediu izolat fara sa ruleze toate celulele de la inceput. Din acest motiv, variabilele globale pot sa nu fie disponibile in timpul evaluarii.


## Cuprins

1.  [Introducere in interpretarea modelelor](#1)
2.  [Interpretarea modelelor liniare](#2)
    *   [Exercitiul 1: Analiza coeficientilor](#ex-1)
3.  [Interpretarea modelelor de tip arbore](#3)
    *   [Exercitiul 2: Vizualizarea arborelui de decizie](#ex-2)
4.  [Metode de importanta a caracteristicilor](#4)
    *   [Exercitiul 3: Importanta prin permutare](#ex-3)
5.  [Aprofundare SHAP](#5)
    *   [Exercitiul 4: Analiza SHAP](#ex-4)
6.  [Explicatii locale cu LIME](#6)
    *   [Exercitiul 5: Explicatii LIME](#ex-5)
7.  [Implementare de la zero](#7)
    *   [Exercitiul 6: Importanta prin permutare de la zero](#ex-6)
8.  [Concluzie](#8)


<a name='1'></a>
## 1. Introducere in interpretarea modelelor

**Interpretarea modelului** reprezinta gradul in care un om poate intelege cauza unei decizii luate de un model de machine learning. Ea raspunde la intrebari precum:
- Care caracteristici conteaza cel mai mult pentru model?
- Cum afecteaza predictia modificarea valorii unei caracteristici?
- De ce a facut modelul *aceasta predictie specifica* pentru *aceasta instanta specifica*?

### Interpretabilitate vs Explicabilitate

| Termen | Domeniu | Exemplu |
|---|---|---|
| **Interpretabilitate** | Global : intelegerea modului in care functioneaza modelul per ansamblu | Importanta caracteristicilor, coeficientii modelului |
| **Explicabilitate** | Local : intelegerea unei singure predictii | Grafic SHAP force, explicatie LIME |

### Tehnici cheie

| Metoda | Domeniu | Independenta de model? | Descriere |
|---|---|---|---|
| Coeficienti | Global | Nu (doar liniar) | Interpretare directa a ponderilor |
| Reguli din arbori | Global/Local | Nu (doar arbori) | Vizualizarea traseului decizional |
| Importanta prin permutare | Global | Da | Scaderea scorului cand caracteristica este amestecata |
| SHAP | Global + Local | Da | Atribuire bazata pe valorile Shapley din teoria jocurilor |
| LIME | Local | Da | Aproximare liniara locala |


### Importa bibliotecile necesare


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import unittests
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, r2_score
from sklearn.base import clone

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Install with: pip install shap")

try:
    from lime import lime_tabular
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("LIME not installed. Install with: pip install lime")

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

### Incarca seturile de date

Vom folosi doua seturi de date bine cunoscute:
- **Breast Cancer** (clasificare, 30 de caracteristici) : prezice tumori maligne vs benigne.
- **Diabetes** (regresie, 10 caracteristici) : prezice progresia bolii la un an dupa momentul initial.


In [ ]:
# ── Classification dataset: Breast Cancer ──────────────────────────────────
cancer = load_breast_cancer()
X_clf = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_clf = cancer.target   # 0 = malignant, 1 = benign

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# ── Regression dataset: Diabetes ────────────────────────────────────────────
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = diabetes.target   # disease progression

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Breast Cancer  : train: {X_clf_train.shape}, test: {X_clf_test.shape}")
print(f"Diabetes       : train: {X_reg_train.shape}, test: {X_reg_test.shape}")

<a name='2'></a>
## 2. Interpretarea modelelor liniare

Modelele liniare (regresie logistica, regresie ridge) sunt inerent interpretabile prin **coeficientii** lor. Fiecare coeficient reprezinta schimbarea iesirii pentru o modificare cu o unitate a caracteristicii corespunzatoare, toate celelalte lucruri ramanand egale.

Pentru caracteristici standardizate, magnitudinea unui coeficient reflecta direct importanta caracteristicii.

### Reguli cheie
- **Coeficient pozitiv** -> o valoare mai mare a caracteristicii impinge predictia in sus.
- **Coeficient negativ** -> o valoare mai mare a caracteristicii impinge predictia in jos.
- **Magnitudinea |coeficientului|** -> forta influentei caracteristicii.
- Trebuie sa **standardizezi** caracteristicile inainte de a compara magnitudinile coeficientilor.


<a name='ex-1'></a>
### Exercitiul 1: Analiza coeficientilor unui model liniar

**Sarcina ta:**

Implementeaza `interpret_linear_model` astfel incat:
1. Sa antreneze un pipeline `Ridge` standardizat pe `(X_train, y_train)`.
2. Sa extraga coeficientii din pipeline-ul antrenat.
3. Sa returneze un `pd.DataFrame` cu coloanele `feature` si `coefficient`, **sortat descrescator dupa valoarea absoluta a coeficientului**.

**Argumente:**
- `X_train`, `y_train` : datele de antrenare (DataFrame + array).
- `X_test`, `y_test` : datele de test pentru calcularea scorului R².
- `alpha` : forta regularizarii Ridge (implicit `1.0`).

**Returneaza:** `(coef_df, r2_score_value)` : DataFrame si float.


In [ ]:
# GRADED FUNCTION: interpret_linear_model
def interpret_linear_model(X_train, y_train, X_test, y_test, alpha=1.0):
    """
    Fit a Ridge regression pipeline (StandardScaler → Ridge) and extract
    sorted coefficient analysis.

    Parameters
    ----------
    X_train : pd.DataFrame
    y_train : array-like
    X_test  : pd.DataFrame
    y_test  : array-like
    alpha   : float, Ridge regularisation strength

    Returns
    -------
    coef_df : pd.DataFrame  columns=['feature', 'coefficient'], sorted by |coef|
    r2      : float         R² on test set
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###
    return coef_df, r2

In [ ]:
# Test interpret_linear_model on Diabetes dataset
coef_df, r2 = interpret_linear_model(
    X_reg_train, y_reg_train, X_reg_test, y_reg_test, alpha=1.0
)

print(f"Ridge R² on test set: {r2:.4f}")
print("\nTop 5 most influential features:")
print(coef_df.head())

# Visualise all coefficients
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0' if c >= 0 else '#C44E52' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Coefficient value (standardised features)')
ax.set_title('Ridge Regression Coefficients : Diabetes Dataset\n'
             '(Blue = positive influence, Red = negative influence)',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_1(interpret_linear_model)

<a name='3'></a>
## 3. Interpretarea modelelor de tip arbore

Arborii de decizie sunt printre cele mai interpretabile modele deoarece fiecare predictie poate fi urmarita ca un traseu printr-o secventa de reguli if/else usor de citit de catre oameni.

Concepte cheie:
- **Importanta caracteristicilor (bazata pe impuritate):** scaderea medie a impuritatii (Gini/entropie) ponderata cu numarul de esantioane care trec prin nod.
- **Traseu de decizie:** secventa de noduri parcursa de o anumita instanta pana la clasa prezisa.
- **`export_text`:** transforma arborele intr-un set de reguli textuale.


<a name='ex-2'></a>
### Exercitiul 2: Vizualizarea arborelui de decizie si extragerea regulilor

**Sarcina ta:**

Implementeaza `interpret_tree_model` astfel incat:
1. Sa antreneze un `DecisionTreeClassifier(max_depth=4, random_state=42)` pe `(X_train, y_train)`.
2. Sa extraga **importanta caracteristicilor** intr-un `pd.DataFrame` cu coloanele `feature` si `importance`, sortat descrescator.
3. Sa extraga regulile textuale folosind `export_text`.
4. Sa calculeze acuratetea pe setul de test.

**Returneaza:** `(importance_df, text_rules, accuracy)`


In [ ]:
# GRADED FUNCTION: interpret_tree_model
def interpret_tree_model(X_train, y_train, X_test, y_test, max_depth=4):
    """
    Fit a DecisionTreeClassifier and extract feature importances and rules.

    Returns
    -------
    importance_df : pd.DataFrame  columns=['feature', 'importance'], sorted desc
    text_rules    : str           human-readable decision rules
    accuracy      : float         test accuracy
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###
    return importance_df, text_rules, accuracy

In [ ]:
# Test interpret_tree_model on Breast Cancer
imp_df, rules, acc = interpret_tree_model(
    X_clf_train, y_clf_train, X_clf_test, y_clf_test, max_depth=4
)

print(f"Decision Tree accuracy: {acc:.4f}")
print("\nTop 5 most important features:")
print(imp_df.head())

print("\nDecision rules (first 600 chars):")
print(rules[:600])

# Plot tree
dt_vis = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_vis.fit(X_clf_train, y_clf_train)

fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt_vis, feature_names=list(X_clf_train.columns),
          class_names=cancer.target_names,
          filled=True, rounded=True, ax=ax, fontsize=8)
ax.set_title('Decision Tree (max_depth=3) : Breast Cancer', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_2(interpret_tree_model)

<a name='4'></a>
## 4. Metode de importanta a caracteristicilor

Metode diferite de importanta spun adesea povesti diferite. Compararea lor scoate la iveala ce caracteristici sunt importante in mod **robust** in mai multe interpretari.

| Metoda | Cum functioneaza | Puncte forte | Limitari |
|---|---|---|---|
| **Impuritate (MDI)** | Scaderea medie a Gini/entropiei | Rapida, integrata | Tinde sa favorizeze caracteristici cu cardinalitate mare |
| **Permutare** | Scaderea scorului dupa amestecarea caracteristicii | Independenta de model, mai putin biased | Lenta, caracteristicile corelate impart importanta |
| **SHAP** | Valori Shapley din teoria jocurilor | Consistenta, aditiva | Cost computational ridicat |


<a name='ex-3'></a>
### Exercitiul 3: Analiza importantei prin permutare

**Sarcina ta:**

Implementeaza `compute_permutation_importance` astfel incat:
1. Sa antreneze un `RandomForestClassifier(n_estimators=100, random_state=42)` pe datele de antrenare.
2. Sa calculeze **importanta prin permutare** pe **setul de test** folosind `sklearn.inspection.permutation_importance` cu `n_repeats=10, random_state=42`.
3. Sa returneze un `pd.DataFrame` cu coloanele `feature`, `importance_mean`, `importance_std`, sortat descrescator dupa `importance_mean`.
4. Sa returneze si acuratetea pe setul de test.

**Returneaza:** `(perm_df, accuracy)`


In [ ]:
# GRADED FUNCTION: compute_permutation_importance
def compute_permutation_importance(X_train, y_train, X_test, y_test):
    """
    Fit RandomForest and compute permutation importance on the test set.

    Returns
    -------
    perm_df  : pd.DataFrame  columns=['feature', 'importance_mean', 'importance_std']
    accuracy : float         test accuracy
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###
    return perm_df, accuracy

In [ ]:
# Test compute_permutation_importance on Breast Cancer
perm_df, rf_acc = compute_permutation_importance(
    X_clf_train, y_clf_train, X_clf_test, y_clf_test
)

print(f"Random Forest accuracy: {rf_acc:.4f}")
print("\nTop 10 features by permutation importance:")
print(perm_df.head(10).to_string(index=False))

# Horizontal bar chart with error bars
fig, ax = plt.subplots(figsize=(10, 7))
top10 = perm_df.head(10)
ax.barh(top10['feature'][::-1], top10['importance_mean'][::-1],
        xerr=top10['importance_std'][::-1],
        color='#4C72B0', alpha=0.85, edgecolor='white', capsize=4)
ax.set_xlabel('Mean accuracy decrease after permutation')
ax.set_title('Permutation Importance (top 10) : Breast Cancer', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_3(compute_permutation_importance)

<a name='5'></a>
## 5. Aprofundare SHAP

**SHAP (SHapley Additive exPlanations)** atribuie fiecarei caracteristici o valoare Shapley : contributia marginala medie a acelei caracteristici in toate subseturile posibile de caracteristici.

### Proprietati cheie
- **Aditivitate:** valorile SHAP pentru toate caracteristicile se aduna exact pana la predictia minus valoarea de baza (`E[f(X)]`).
- **Consistenta:** daca un model se schimba astfel incat o caracteristica are un impact mai mare, valoarea sa SHAP creste.
- **Acuratete locala:** `f(x) = base_value + sum(shap_values)`.

### Tipuri de grafice SHAP
| Grafic | Scop |
|---|---|
| **Summary plot** | Beeswarm care arata distributia impactului caracteristicilor pentru toate esantioanele |
| **Bar plot** | Valoarea SHAP absoluta medie pe caracteristica (importanta globala) |
| **Dependence plot** | Valoarea SHAP vs valoarea caracteristicii (arata neliniaritatile) |
| **Force plot** | Descompunerea unei singure predictii in contributiile caracteristicilor |


<a name='ex-4'></a>
### Exercitiul 4: Analiza SHAP

**Sarcina ta:**

Implementeaza `compute_shap_values` astfel incat:
1. Sa antreneze un `GradientBoostingClassifier(n_estimators=100, random_state=42)` pe datele de antrenare.
2. Sa creeze un `shap.TreeExplainer` pentru modelul antrenat.
3. Sa calculeze valorile SHAP pentru `X_test`.
4. Sa returneze `(explainer, shap_values, accuracy)`.

**Nota:** daca `shap` nu este instalat, returneaza `(None, None, accuracy)` astfel incat restul notebook-ului sa poata rula in continuare.


In [ ]:
# GRADED FUNCTION: compute_shap_values
def compute_shap_values(X_train, y_train, X_test, y_test):
    """
    Fit GradientBoostingClassifier and compute SHAP values using TreeExplainer.

    Returns
    -------
    explainer   : shap.TreeExplainer  (or None if shap not available)
    shap_values : np.ndarray or list  SHAP values for X_test
    accuracy    : float               test accuracy
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###
    return explainer, shap_values, accuracy

In [ ]:
# Test compute_shap_values
explainer, shap_vals, gb_acc = compute_shap_values(
    X_clf_train, y_clf_train, X_clf_test, y_clf_test
)
print(f"Gradient Boosting accuracy: {gb_acc:.4f}")

if SHAP_AVAILABLE and shap_vals is not None:
    print(f"SHAP values shape: {np.array(shap_vals).shape}")

    # Summary beeswarm plot
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_vals, X_clf_test, show=False)
    plt.title('SHAP Summary Plot : Breast Cancer (Gradient Boosting)', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Bar plot: mean |SHAP|
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_clf_test, plot_type='bar', show=False)
    plt.title('Mean |SHAP| Feature Importance', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Dependence plot for top feature
    top_feat = pd.DataFrame({
        'feature': list(X_clf_test.columns),
        'mean_shap': np.abs(shap_vals).mean(0)
    }).sort_values('mean_shap', ascending=False).iloc[0]['feature']
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(top_feat, shap_vals, X_clf_test, show=False)
    plt.title(f'SHAP Dependence Plot : {top_feat}', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("SHAP not available : skipping plots. Install with: pip install shap")

In [ ]:
unittests.exercise_4(compute_shap_values)

<a name='6'></a>
## 6. Explicatii locale cu LIME

**LIME (Local Interpretable Model-agnostic Explanations)** explica predictiile individuale prin antrenarea unui model mai simplu si interpretabil (liniar) in **vecinatatea locala** a instantei explicate.

### Cum functioneaza LIME
1. Selectezi instanta pe care vrei sa o explici.
2. Generezi **esantioane perturbate** in jurul ei.
3. Obtii predictiile modelului pentru toate esantioanele perturbate.
4. Ponderezi esantioanele in functie de **proximitatea** lor fata de instanta originala.
5. Antrenezi un **model liniar ponderat** pe esantioanele perturbate.
6. Coeficientii modelului liniar reprezinta explicatia.

### Global vs Local
| SHAP | LIME |
|---|---|
| Atribuire exacta bazata pe teoria jocurilor | Aproximare liniara locala |
| Consistent la nivel global | Valabil doar local |
| Mai lent (pentru arbori: rapid, pentru retele neuronale: lent) | Rapid pentru orice model |
| Poate surprinde interactiuni intre caracteristici | Presupune liniaritate locala |


<a name='ex-5'></a>
### Exercitiul 5: Explicatii LIME

**Sarcina ta:**

Implementeaza `compute_lime_explanation` astfel incat:
1. Sa creeze un `LimeTabularExplainer` folosind `X_train` (numpy array), `feature_names`, `class_names` si `mode='classification'`.
2. Sa genereze o explicatie pentru instanta `X_test.iloc[instance_idx]` folosind `num_features=10`.
3. Sa extraga explicatia ca o lista de tupluri `(feature_name, weight)` sortata **descrescator dupa valoarea absoluta a ponderii**.
4. Sa returneze `(explanation_list, lime_explainer)`.

**Returneaza:** `([(feature, weight), ...], explainer_object)`


In [ ]:
# GRADED FUNCTION: compute_lime_explanation
def compute_lime_explanation(model, X_train, X_test, feature_names,
                              class_names, instance_idx=0):
    """
    Generate a LIME explanation for a single instance.

    Parameters
    ----------
    model         : fitted sklearn classifier
    X_train       : pd.DataFrame  training data (used to build the explainer)
    X_test        : pd.DataFrame  test data
    feature_names : list of str
    class_names   : list of str
    instance_idx  : int           index of the test instance to explain

    Returns
    -------
    explanation_list : list of (feature_name, weight) sorted by |weight| desc
    lime_explainer   : LimeTabularExplainer object (or None if LIME unavailable)
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###
    return explanation_list, lime_explainer

In [ ]:
# Fit a GradientBoosting model for LIME
gb_lime = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_lime.fit(X_clf_train, y_clf_train)

instance_idx = 5   # explain the 6th test instance

lime_list, lime_exp_obj = compute_lime_explanation(
    model         = gb_lime,
    X_train       = X_clf_train,
    X_test        = X_clf_test,
    feature_names = list(X_clf_train.columns),
    class_names   = list(cancer.target_names),
    instance_idx  = instance_idx
)

true_label = y_clf_test.iloc[instance_idx] if hasattr(y_clf_test, 'iloc') else y_clf_test[instance_idx]
pred_label = gb_lime.predict(X_clf_test.iloc[[instance_idx]])[0]
print(f"Instance {instance_idx}: true={cancer.target_names[true_label]}, "
      f"predicted={cancer.target_names[pred_label]}")

if lime_list:
    print("\nTop LIME feature weights:")
    for feat, w in lime_list[:6]:
        print(f"  {feat:40s} {w:+.4f}")

    # Bar chart of LIME weights
    feats_l = [x[0] for x in lime_list]
    weights  = [x[1] for x in lime_list]
    colors_l = ['#55A868' if w > 0 else '#C44E52' for w in weights]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(feats_l[::-1], weights[::-1], color=colors_l[::-1],
            edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=1.2)
    ax.set_xlabel('LIME weight')
    ax.set_title(f'LIME Local Explanation : Instance {instance_idx}\n'
                 f'True: {cancer.target_names[true_label]}, '
                 f'Predicted: {cancer.target_names[pred_label]}',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("LIME not available : skipping.")

In [ ]:
unittests.exercise_5(compute_lime_explanation)

<a name='7'></a>
## 7. Implementare de la zero

Intelegerea algoritmului din spatele importantei prin permutare iti aprofundeaza intuitia si iti permite sa il adaptezi la scenarii nestandard (metrici personalizate, permutari pe grupuri etc.).


<a name='ex-6'></a>
### Exercitiul 6: Importanta prin permutare de la zero

**Sarcina ta:**

Implementeaza `permutation_importance_scratch` astfel incat:
1. Sa calculeze scorul de baza `score(model, X_test, y_test)` folosind `scoring_fn` furnizat.
2. Pentru fiecare coloana de caracteristica (dupa index), sa amestece **doar acea coloana** intr-o copie a lui `X_test` (toate celelalte raman neschimbate).
3. Sa calculeze scorul pe datele amestecate.
4. Sa inregistreze `importance = baseline_score - shuffled_score`.
5. Sa returneze un `pd.DataFrame` cu coloanele `feature`, `importance`, sortat descrescator.

**Argumente:**
- `model` : estimator antrenat.
- `X_test` : caracteristicile de test de tip `pd.DataFrame`.
- `y_test` : etichetele reale.
- `scoring_fn` : functia apelabila `(model, X, y) -> float`, de exemplu `accuracy_score`.
- `random_state` : int, pentru reproductibilitate.

**Returneaza:** `pd.DataFrame` cu coloanele `['feature', 'importance']`


In [ ]:
# GRADED FUNCTION: permutation_importance_scratch
def permutation_importance_scratch(model, X_test, y_test, scoring_fn, random_state=42):
    """
    Compute permutation importance from scratch.

    Parameters
    ----------
    model       : fitted estimator with .predict()
    X_test      : pd.DataFrame
    y_test      : array-like
    scoring_fn  : callable (model, X, y) -> float
    random_state: int

    Returns
    -------
    pd.DataFrame  columns=['feature', 'importance'], sorted descending
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###
    return result

In [ ]:
# Test permutation_importance_scratch
# Use the Random Forest trained in Exercise 3
rf_ex6 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ex6.fit(X_clf_train, y_clf_train)

def acc_fn(mdl, X, y):
    return accuracy_score(y, mdl.predict(X))

scratch_df = permutation_importance_scratch(
    rf_ex6, X_clf_test, y_clf_test, scoring_fn=acc_fn, random_state=42
)

# Compare scratch vs sklearn
sklearn_result = permutation_importance(
    rf_ex6, X_clf_test, y_clf_test, n_repeats=1, random_state=42
)
sklearn_df = pd.DataFrame({
    'feature':    list(X_clf_test.columns),
    'sk_importance': sklearn_result.importances_mean
}).sort_values('sk_importance', ascending=False).reset_index(drop=True)

# Rank correlation between methods
from scipy.stats import spearmanr
merged = scratch_df.merge(sklearn_df, on='feature')
corr, pval = spearmanr(merged['importance'], merged['sk_importance'])
print(f"Spearman rank correlation (scratch vs sklearn): {corr:.4f}  (p={pval:.4f})")

print("\nTop 10 by scratch:")
print(scratch_df.head(10).to_string(index=False))

# Side-by-side bars
top10_s = scratch_df.head(10)
top10_k = sklearn_df.head(10)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df_, title, col in [
        (axes[0], top10_s, 'Permutation Importance (Scratch)', '#4C72B0'),
        (axes[1], top10_k, 'Permutation Importance (sklearn)', '#55A868')]:
    ax.barh(df_['feature'].iloc[::-1],
            df_.iloc[::-1, 1],
            color=col, alpha=0.85, edgecolor='white')
    ax.set_xlabel('Importance (accuracy drop)')
    ax.set_title(title, fontweight='bold')
plt.suptitle('Scratch vs sklearn Permutation Importance Comparison',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_6(permutation_importance_scratch)

<a name='8'></a>
## 8. Concluzie

**Felicitari!** Ai finalizat tema despre Interpretarea si Explicabilitatea Modelelor.

### Idei principale

| Exercitiu | Tehnica | Ce ai invatat |
|---|---|---|
| 1 | Coeficienti Ridge | Coeficientii standardizati arata direct directia si magnitudinea impactului caracteristicilor |
| 2 | Reguli din arborele de decizie | Fiecare predictie poate fi urmarita ca un traseu if/else usor de citit |
| 3 | Importanta prin permutare | Importanta independenta de model, calculata pe setul de test, care evita biasul MDI |
| 4 | SHAP TreeExplainer | Valori Shapley exacte si aditive pentru explicatii globale + locale |
| 5 | LIME | Aproximare liniara locala rapida pentru orice model black-box |
| 6 | Permutare (de la zero) | Algoritmul de baza: amesteca o caracteristica si masoara scaderea scorului |

### Cand folosesti fiecare metoda

```
Trebuie sa explici modelul global echipei?
  -> SHAP bar plot (mean |SHAP value|) sau importanta prin permutare

Trebuie sa explici o singura predictie (de ex. respingerea unui credit)?
  -> SHAP force plot sau explicatie LIME

Lucrezi cu un model liniar?
  -> Coeficientii standardizati sunt exacti si suficienti

Ai nevoie de importante rapide, independente de model?
  -> Importanta prin permutare (1 repetare) : secunde, nu minute
```

### Nota despre instalare

Daca SHAP sau LIME nu sunt disponibile, instaleaza-le:
```bash
pip install shap lime
```
